# JAXA Earth API で学ぶ衛星データ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astrocamp-2026-siaPpts/astrocamp-sat-gis/blob/main/public/notebooks/tutorials/jaxa_intro.ipynb)

このノートブックでは、JAXA Earth API for Python を使って日本の衛星データに触れます。
**認証不要** で使えるため、GEE アカウントがない方でもすぐに始められます。

### 学習目標
1. JAXA Earth API の基本的な使い方を理解する
2. ALOS 標高データで地形を可視化できる
3. MODIS NDVI で植生の時系列変化を確認できる
4. JAXA API と GEE の使い分けを説明できる

### GEE との違い
| 項目 | GEE (Google Earth Engine) | JAXA Earth API |
|------|--------------------------|----------------|
| 認証 | Cloud Project 登録必須 | **不要** |
| 光学画像 | Sentinel-2 (10m) など豊富 | MODIS (250m〜1km) 中心 |
| 強み | 高解像度光学・多種データ | 日本の衛星（ALOS, GSMaP）に特化 |
| 構文 | `ee.ImageCollection().filter()` | `je.ImageCollection().filter()` |

> **位置づけ:** JAXA API は GEE の代替ではなく補完です。標高・降水など日本の衛星データを手軽に取得したいときに使います。

In [ ]:
# 必要なライブラリのインストール（Colab 環境の場合）
!pip install jaxa-earth

In [ ]:
from jaxa.earth import je
import matplotlib.pyplot as plt
import numpy as np
import sys

# ── matplotlib 日本語フォント設定 ──
import matplotlib.font_manager as _fm
import platform
_IN_COLAB = 'google.colab' in sys.modules
if _IN_COLAB:
    import subprocess
    subprocess.run(['apt-get', 'install', '-y', 'fonts-noto-cjk'], capture_output=True)
    _fm._load_fontmanager(try_read_cache=False)
_system = platform.system()
_jp_candidates = {
    'Darwin': ['Hiragino Sans', 'AppleGothic'],
    'Windows': ['Yu Gothic', 'MS Gothic'],
}.get(_system, ['Noto Sans CJK JP', 'IPAexGothic'])
_found = [f for f in _jp_candidates if f in [x.name for x in _fm.fontManager.ttflist]]
if _found:
    plt.rcParams['font.sans-serif'] = _found
else:
    _alt = sorted(set(x.name for x in _fm.fontManager.ttflist
                       if any(c in x.name for c in ['Noto', 'IPA', 'Gothic', 'Yu', 'Hiragino'])))
    if _alt:
        plt.rcParams['font.sans-serif'] = _alt
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False

print('準備完了')

---
## 解析対象範囲

福島県浜通り（南相馬市周辺）を対象とします。
GEE のノートブックと同じ範囲です。

In [ ]:
# 解析対象範囲（南相馬市周辺）
bbox = [139.9, 37.2, 140.6, 37.8]  # [min_lon, min_lat, max_lon, max_lat]

print(f'範囲: 経度 {bbox[0]}〜{bbox[2]}, 緯度 {bbox[1]}〜{bbox[3]}')

---
## 利用可能なデータセットを検索する

`ImageCollectionList.filter_name()` でキーワード検索ができます。
例えば「標高」「NDVI」「降水」に関連するデータセットを探してみましょう。

In [ ]:
# キーワードでデータセットを検索
keywords = ['ndvi', 'monthly']
collections, bands = je.ImageCollectionList(ssl_verify=True).filter_name(keywords=keywords)

print(f'キーワード {keywords} に一致: {len(collections)} 件\n')
for i, (col, bnd) in enumerate(zip(collections, bands)):
    print(f'[{i}] {col}')
    print(f'    バンド: {bnd}\n')

---
## ALOS 標高データ（DEM）を表示する

JAXA の **ALOS AW3D30** は、全球を 30m 解像度でカバーする数値標高モデルです。
福島県浜通りの地形を可視化してみましょう。

In [ ]:
# ALOS AW3D30 から標高データを取得
print('ALOS DEM を取得中...')

collection = 'JAXA.EORC_ALOS.PRISM_AW3D30.v3.2_global'
band = 'DSM'
dlim = ['2021-02-01T00:00:00', '2021-02-01T00:00:00']
ppu = 360  # 約 300m/pixel (bbox 全体をカバー)

data = (
    je.ImageCollection(collection=collection, ssl_verify=True)
    .filter_date(dlim=dlim)
    .filter_resolution(ppu=ppu)
    .filter_bounds(bbox=bbox)
    .select(band=band)
    .get_images()
)

# numpy 配列を取得
img = je.ImageProcess(data).show_images()
dem_array = img.raster.img[0]

print(f'配列形状: {dem_array.shape}')
print(f'標高範囲: {np.nanmin(dem_array):.0f}〜{np.nanmax(dem_array):.0f} m')

In [ ]:
# 標高データを可視化
plt.figure(figsize=(12, 8))
plt.imshow(dem_array, cmap='terrain', aspect='auto')
cbar = plt.colorbar(shrink=0.8, label='標高 (m)')
plt.title('ALOS AW3D30 標高データ — 福島県浜通り', fontsize=14)
plt.xlabel('ピクセル (x)')
plt.ylabel('ピクセル (y)')
plt.grid(False)
plt.show()

> **考察:** 地形の特徴を観察しましょう。
> - 西側（左）の阿武隈山地と東側（右）の太平洋岸で標高がどう変化するか？
> - 南相馬市の農地（中央やや右上）はどのような地形に位置しているか？
> - この標高データは SAR 解析（Week 2-3）の地形補正に使えます。

---
## MODIS NDVI で植生を確認する

MODIS 衛星の NDVI データ（250m 解像度）を表示します。
月別の NDVI と平年値（2012-2021年平均）の差分を見てみましょう。

In [ ]:
# MODIS NDVI（月別）を取得
print('MODIS NDVI を取得中...')

collection_ndvi = 'JAXA.G-Portal_GCOM-C.SGLI_standard.L3-NDVI.daytime.v3_global_monthly'
band_ndvi = 'NDVI'
dlim_ndvi = ['2021-08-01T00:00:00', '2021-08-01T00:00:00']
ppu_ndvi = 180  # 約 600m/pixel

data_ndvi = (
    je.ImageCollection(collection=collection_ndvi, ssl_verify=True)
    .filter_date(dlim=dlim_ndvi)
    .filter_resolution(ppu=ppu_ndvi)
    .filter_bounds(bbox=bbox)
    .select(band=band_ndvi)
    .get_images()
)

img_ndvi = je.ImageProcess(data_ndvi).show_images()
ndvi_array = img_ndvi.raster.img[0]
print(f'配列形状: {ndvi_array.shape}')
print(f'NDVI 範囲: {np.nanmin(ndvi_array):.3f}〜{np.nanmax(ndvi_array):.3f}')

In [ ]:
# NDVI を可視化
plt.figure(figsize=(12, 8))
plt.imshow(ndvi_array, cmap='RdYlGn', vmin=-0.5, vmax=1.0, aspect='auto')
cbar = plt.colorbar(shrink=0.8, label='NDVI')
plt.title('MODIS NDVI (2021年8月) — 福島県浜通り', fontsize=14)
plt.xlabel('ピクセル (x)')
plt.ylabel('ピクセル (y)')
plt.show()

> **考察:**
> - 高い NDVI（緑）はどこに分布しているか？
> - 水域（太平洋）の NDVI はどのような値か？
> - GEE で見た Sentinel-2 NDVI と比べて、解像度の違いを実感しよう。

---
## （参考）時系列統計を計算する

JAXA API は時系列データの空間統計も計算できます。
年単位の NDVI 平均値・標準偏差などをグラフ化してみましょう。

In [ ]:
# 複数日付の空間統計を計算
print('時系列統計を計算中（数分かかる場合があります）...')

dlim_ts = ['2021-01-01T00:00:00', '2021-12-31T00:00:00']

data_ts = (
    je.ImageCollection(collection=collection_ndvi, ssl_verify=True)
    .filter_date(dlim=dlim_ts)
    .filter_resolution(ppu=ppu_ndvi)
    .filter_bounds(bbox=bbox)
    .select(band=band_ndvi)
    .get_images()
)

try:
    img_ts = (
        je.ImageProcess(data_ts)
        .calc_spatial_stats()
        .show_spatial_stats()
    )
except Exception as e:
    print(f'時系列計算でエラー: {e}')
    print('複数日付の処理には時間がかかる場合があります。')

---
## GSMaP 降水量データ

**GSMaP**（Global Satellite Mapping of Precipitation）は、JAXA が提供する全球降水マップです。
福島の梅雨期の降水パターンを確認してみましょう。

In [ ]:
# GSMaP 降水量平年値（半月別）を取得
print('GSMaP 降水量を取得中...')

collection_rain = 'JAXA.EORC_GSMaP_standard.Gauge.00Z-23Z.v6_half-monthly-normal'
band_rain = 'PRECIP_2012_2021'
dlim_rain = ['2021-07-01T00:00:00', '2021-07-01T00:00:00']
ppu_rain = 180

data_rain = (
    je.ImageCollection(collection=collection_rain, ssl_verify=True)
    .filter_date(dlim=dlim_rain)
    .filter_resolution(ppu=ppu_rain)
    .filter_bounds(bbox=bbox)
    .select(band=band_rain)
    .get_images()
)

img_rain = je.ImageProcess(data_rain).show_images()
rain_array = img_rain.raster.img[0]

print(f'配列形状: {rain_array.shape}')
print(f'降水範囲: {np.nanmin(rain_array):.1f}〜{np.nanmax(rain_array):.1f} mm/半月')

In [ ]:
# 降水量を可視化
plt.figure(figsize=(12, 8))
plt.imshow(rain_array, cmap='Blues', aspect='auto')
cbar = plt.colorbar(shrink=0.8, label='降水量 (mm/半月)')
plt.title('GSMaP 降水量平年値 (7月上旬) — 福島県浜通り', fontsize=14)
plt.xlabel('ピクセル (x)')
plt.ylabel('ピクセル (y)')
plt.show()

> **考察:**
> - 沿岸部と内陸部で降水量に差はあるか？
> - この降水データと GEE の Sentinel-1 SAR データを組み合わせると、
>   降水後の土壌水分変化を追跡できます（発展テーマ）。

---
## まとめ

### JAXA Earth API の特徴

| メリット | デメリット |
|---------|-----------|
| 認証不要で即座に使える | 光学画像の解像度が低い（MODIS 250m〜） |
| 日本の衛星データ（ALOS, GSMaP, GCOM-W）が充実 | データ種類は GEE より限定的 |
| matplotlib でそのまま表示可能 | 複数日付の時系列処理が遅い |

### 使い分けの指針

| やりたいこと | 使う API |
|------------|---------|
| 高解像度光学画像（Sentinel-2）で NDVI | **GEE** |
| 標高データ（DEM）の取得 | JAXA (ALOS) / どちらでも可 |
| 降水データ（GSMaP） | **JAXA**（GSMaP は JAXA 独自） |
| SAR データ（Sentinel-1 / ALOS-2） | **GEE**（JAXA API に SAR なし） |
| アカウント登録なしで手軽に試す | **JAXA** |

### 関連リンク
- [JAXA Earth API 公式ドキュメント](https://data.earth.jaxa.jp/api/python/v0.1.6/ja/)
- [データセット一覧](https://data.earth.jaxa.jp/en/datasets/)
- [API リファレンス](https://data.earth.jaxa.jp/api/python/v0.1.6/ja/api-reference/)